# Word embeddings for document (review) classification

This notebook exemplifies text classification using a pre-trained word embeddings model ([GloVe](https://nlp.stanford.edu/projects/glove/)) for transforming textual data into numerical vectors (to serve as the input for classification algorithms).<br>

The dataset used in the example is the same as the one used in the previous two classes. It originates from the paper [Listening to the Crowd for the Release Planning of Mobile Apps](https://ieeexplore.ieee.org/abstract/document/8057860) and is available from [this web page](https://dibt.unimol.it/report/others/clap/) with the supplementary materials for the paper.

### The initial setup: downloading the required models, loading packages, setting up some useful variables

Up to now we’ve been using spaCy’s smallest English language pipeline (_en_core_web_sm_), which provides vocabulary, syntax, and entities, but not vectors. To make advantage of the spaCy's built-in word vector model we’ll need a larger pipeline, namely either _en_core_web_md_ or _en_core_web_lg_. An overview of all the NLP piplines is available at [https://spacy.io/models/en](https://spacy.io/models/en)

In the computer lab we will use the medium model (as that one is available), but you are advised to download and work on your computer with the large spaCy's english language pipeline (`en_core_web_lg`). To download it, run the following line in Terminal or Command Prompt:<br>
`python -m spacy download en_core_web_lg`

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

# modules for oversampling and pipeline building
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn import metrics
from sklearn.metrics import ConfusionMatrixDisplay

# sklearn modules that implement classification algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import spacy
from pathlib import Path

RAND_STATE = 1

# spacy pipeline to use for text preprocessing and word vectors
SPACY_MODEL = "en_core_web_md"

### Explore spaCy's pre-trained word embeddings model (GloVe)

Load the spaCy pipeline with pre-trained GloVe word embeddings (vectors):

The loaded NLP pipeline includes 20K unique vectors for 685K distinct words in case of the medium model and 343K unique vectors for 685K words in case of the large model; in both cases words are represented with 300-dimensional vectors.

A note regarding the discrepancy in number of words (685K) and the number of unique vectors (21K in medium and 343K in large model): to reduce the model size, spacy maps several closely related words to the same vector; for example, words such as 'kid', 'kids', 'child', 'children' would all be mapped to the same vector.

Let's see what a word embedding looks like:

#### Word similarity

Having words represented as vectors, we can use linear algebra and vector space models to analyze the relationship between words.

We will start by examining word similarity, by computing the ***cosine similarity*** of word vectors. Cosine similarity is a widely used measure of semantic similarity of words. Its value ranges from -1 to 1, though it is usually used in the non-negative space [0, 1] where 0 means absence of similarity and 1 means extremely similar or identical.

Cosine similarity is implemented in the spacy's `similarity` function

#### Visualization of Word Embeddings

Visualisation is often used to get a better understanding of how word vectors model relations among words.

Since word vectors are high dimensional - GloVe has 300 dimensions and more recent models have a few thousand dimensions - we cannot visualize them directly.
A typical approach to overcome that challenge is to apply dimensionality reduction algorithm and visualize thus transformed data.
Next time, we will use [t-SNE](https://lvdmaaten.github.io/tsne/), a popular dimension reduction technique to reduce the word vectors to 2D and explore the word relations to see if we can find some pattern visually.
<br>
For now, we will explore an interactive visualization of word embeddings can be found here:
[https://projector.tensorflow.org/](https://projector.tensorflow.org/)

## Use word embeddings for text classification

### Load and prepare the data

We use the mobile apps reviews dataset that was downloaded from the above-linked web page and stored locally.

In [ ]:
# code to upload the data file if running the notebook in Google Colab
#
# from google.colab import files
#
# data_files = files.upload()
# data_file_path = list(data_files.keys())[0]

We will re-code the outcome variable as in the previous classes, in particular:

* functional: BUG, FEATURE
* non-functional: PERFORMANCE, USABILITY, ENERGY, SECURITY
* irrelevant: OTHER


Add review length and keep the review, review length, and rating columns (as in the previous classes), then split the data set into training and test sets

### Use word embeddings for document representation

We have a way to numerically represent each word - the embeddings provide a vector for every word in our vocabulary. However, to do the classification, we need to have a numerical representation of complete reviews (documents). So, for each review, we need a vector that would represent that review and that vector should be based on embeddings of the words in the review. <br>
There are several options to do that; here, we apply a very simple method: we average the embeddings of the words forming the review.
Though simple, this method often works well for short texts such as the reviews we are working with.

#### Creating a custom transformer class

We will create a class - `ReviewVectorizer` - that transforms every review into a vector representation, as described above, using a pre-trained GloVe model (integrated in the spaCy NLP pipeline). This transformation is preceded by reducing the text to semantics-bearing words, so that only 'the most important' words are used for building a text (review) representation.

Note that the `ReviewVectorizer` class is defined as a subclass of sklearn's `BaseEstimator` and `TransformerMixin`. This is required in order to make our transformer component compatible with the sklearn's tools such as GridSearchCV and Pipeline. In particular, by subclassing `TransformerMixin`, our class becomes a sklearn Transformer and thus eligible for inclusion in a Pipeline. To allow for setting custom parameters (attributes) for our transformer via the Pipeline, we also need to subclass `BaseEstimator` as that will provide us with `set_param` and `get_param` methods that allow for setting and getting the transformer's parameters (attributes) during the runtime.

When creating a custom transformer by inheriting from `BaseEstimator` and `TransformerMixin`, we are primarily required to implement the following methods:
* `__init__(self, *args, **kwargs)`. The `__init__`'s role is to initialize the transformer. **Important**: all parameters (attributes) of our custom transformer that can be set by the user must be defined as the input arguments of the `__init__` method and stored as instance attributes.
* `fit(self, X, y=None)`. The purpose of this method is to learn any necessary parameters from the training data, $X$; for a transformer, this usually involves calculating statistics like the mean or standard deviation. It must return self.
* `transform(self, X)`. The role of this method is to apply the transformation to the input data, $X$, using the parameters learned in `fit`. It must return the transformed data.

If interested to learn more, the sklearn's documentation (in particular, [this page](https://scikit-learn.org/stable/developers/develop.html#rolling-your-own-estimator)) provides detailed information.

### Train classifiers on the document embeddings

Use the `ReviewVectorizer` class to transform reviews into vectors using a word embeddings model.

Use [`RobustScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html) to rescale the two additional features (using median and inter-quartile range) so that all the features are in the roughly equal value range.

Then we can use the resulting feature set to build a classifier.

We will use **Logistic Regression** as a classifier, as classifiers based on linear boundary are suitable for the vector space that dense (embedding) vectors form.

We are also using [SMOTE resampling method](https://www.analyticsvidhya.com/blog/2020/10/overcoming-class-imbalance-using-smote-techniques/) to address the problem of imbalanced dataset (recall that the three review categories (classes) are not equally present).

**SMOTE** is abbreviation from *Synthetic Minority Oversampling Technique*, and is a technique for increasing the size of a minority class (and thus balancing the class distribution) by synthesizing new instances of that class from the existing instances. In short, SMOTE works as follows:

A random instance from the minority class is first chosen. Then k of the nearest neighbors for that instance are found (typically k=5). A randomly selected neighbor is chosen and a synthetic instance is created at a randomly selected point between the two instances in the feature space.

Finally, note that we are using Pipeline from the `imblearn.pipeline` (not from `sklearn.pipeline` as in the previous classes), since this kind of pipeline properly handles class balancing methods (apply them to the train set, but not to the test set)

#### Evaluate the pipeline

This is not much better than the performance we've had with the BoW model.
Fine-tuning of the pipeline might help improve the results - try it out on your own.

We will explore if another classification algorithm - one often used in text classfication - can lead to better performance.

### Create and evaluate an SVM classifier

This time, we will use a Support Vector Machines (SVM) classifier. This type of classifier has proven successful in text classification tasks; it is generally known to deal well with high dimensional data (as is the case here). It also works well with (relatively) small datasets.
For a quick introduction to SVM, watch this short [video tutorial](https://www.youtube.com/watch?v=N1vOgolbjSc).

We will use the simplest version of this classifier, which is SVM with linear kernel (that is, linear transformation of the original features). This type of classifier can be implemented using sklearn's [SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC) class with the 'linear' kernel.

Make predictions on the test set and compute evaluation measures

### Improve the SVM model through hyperparameter tuning

We can try to improve the classification model through hyperparameter tuning. In particular, we'll examine a range of values for the hyperparameter **C**. With this hyperparameter, we specify how much we want to penalize missclassified points - the higher **C** value, the higher the penalty, but also the higher risk of overfitting. We will do the tuning through grid search over the specified range of parameter C values.<br>

Reminder: for GridSearchCV, hyperparamter names have to be preceded by the name of the classifier as given in the pipeline; in this case, the name is `classifer`, so the hyperparamter C gets prefix `classifier__`.

Re-train the SVM classifier with the optimal C value

Evaluate the optimised model

### Compare all the models

Auxiliary function for extracting evaluation measures from `metrics.classification_report` function

Create also a dummy classifier to serve as the general baseline.

Compare all the models using the above defined `get_model_metrics` auxiliary function